In [1]:
from __future__ import annotations

import io
import fitz
from PIL import Image
from pathlib import Path
from pprint import pprint
from collections.abc import Mapping
from IPython.display import display

from statistics import median
from dataclasses import dataclass

from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

In [14]:
import json
from pathlib import Path
from docling.document_converter import DocumentConverter
from docling.datamodel.document import DoclingDocument
from docling_core.types.doc.document import PictureItem, TableItem, ListItem, CodeItem, FormulaItem

In [3]:
def load_docling_document(pdf_path: str | Path) -> DoclingDocument:
    """
    Load a DoclingDocument from cache if available.
    Otherwise, convert the PDF and create the cache automatically.
    """

    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    # Auto-generate cache path
    cache_path = pdf_path.with_suffix(".docling.json")

    # Load from cache
    if cache_path.exists():
        print(f"✅ Cache found. Loading cached document: {cache_path}")
        doc = DoclingDocument.load_from_json(cache_path)
        print(f"✅ Cache loaded as DoclingDocument")

    # Convert PDF and create cache
    else:
        print("⚠️ Cache not found.")
        print("⏳ Converting PDF using Docling DocumentConverter...")

        converter = DocumentConverter()
        result = converter.convert(pdf_path)
        doc = result.document

        # Save cache for future reuse
        doc.save_as_json(cache_path)

        print(f"✅ Conversion complete.")
        print(f"💾 Cache saved to: {cache_path}")

    # -----------------------------
    # Print document metadata
    # -----------------------------
    print("\n📄 Document Metadata")
    print("-" * 40)

    try:
        print(f"File Name      : {pdf_path.name}")
        print(f"Total Pages    : {len(doc.pages)}")
        print(f"Total Tables   : {len(doc.tables)}")
        print(f"Total Pictures : {len(doc.pictures)}")

        # Optional additional stats
        if hasattr(doc, "texts"):
            print(f"Text Blocks    : {len(doc.texts)}")

    except Exception as e:
        print(f"⚠️ Could not extract full metadata: {e}")

    print("-" * 40)

    return doc

In [4]:
# Example usage
sci_pdf_path = (
    "/mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/"
    "iid-platform/sample_data/named/"
    "book-ch-on-axis-sun-tracking-system.pdf"
)
sci_doc = load_docling_document(sci_pdf_path)
sci_fitz_pdf = fitz.open(sci_pdf_path)

⚠️ Cache not found.
⏳ Converting PDF using Docling DocumentConverter...


[INFO] 2026-05-27 17:51:47,589 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 17:51:47,675 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 17:51:47,676 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 17:51:48,064 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 17:51:48,084 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-05-27 17:51:48,085 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer

✅ Conversion complete.
💾 Cache saved to: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/iid-platform/sample_data/named/book-ch-on-axis-sun-tracking-system.docling.json

📄 Document Metadata
----------------------------------------
File Name      : book-ch-on-axis-sun-tracking-system.pdf
Total Pages    : 30
Total Tables   : 1
Total Pictures : 19
Text Blocks    : 990
----------------------------------------


In [5]:
dict(sci_doc).keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'field_regions', 'field_items', 'pages'])

In [7]:
table0 = sci_doc.tables[0]
df0 = table0.export_to_dataframe()
# df0

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


In [9]:
table = sci_doc.tables[0]
def r(x):
    if isinstance(x, Mapping):
        return {k: r(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [r(v) for v in x]
    try:
        return r(dict(x))
    except:
        return x

# pprint(r(table))

In [12]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

In [15]:
# 1. Configure the pipeline to enable code and formula detection
pipeline_options = PdfPipelineOptions()
pipeline_options.do_code_enrichment = True
pipeline_options.do_formula_enrichment = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# 2. Convert the document
result = converter.convert(sci_pdf_path)
doc = result.document

# 3. Access CodeBlocks and Formulas
for item, _ in doc.iterate_items():
    if isinstance(item, CodeItem):
        print(f"Code ({item.code_language}): {item.text}")
    elif isinstance(item, FormulaItem):
        print(f"Formula (LaTeX): {item.text}")


[INFO] 2026-05-27 19:12:03,050 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 19:12:03,527 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 19:12:03,529 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 19:12:03,797 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 19:12:03,837 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-05-27 19:12:03,838 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer

Formula (LaTeX): S = \begin{bmatrix} S _ { M } \\ S _ { E } \\ S _ { P } \end{bmatrix} = \begin{bmatrix} \cos \delta \cos \omega \\ - \cos \delta \sin \omega \\ \sin \delta \end{bmatrix}
Formula (LaTeX): \delta = \sin ^ { 1 } \{ 0 . 3 9 7 9 5 \cos \left [ 0 . 9 8 5 6 3 \left ( N - 1 7 3 \right ) \right ] \} \ \text {degrees} \right )
Formula (LaTeX): \omega = 1 5 ( t _ { s } - 1 2 ) \quad ( \text {degrees} )
Formula (LaTeX): [ \Phi ] = \begin{bmatrix} \cos \Phi & 0 & \sin \Phi \\ 0 & 1 & 0 \\ - \sin \Phi & 0 & \cos \Phi \end{bmatrix}
Formula (LaTeX): S ^ { \prime } = \begin{bmatrix} S _ { V } \\ S _ { H } \end{bmatrix} = \begin{bmatrix} \sin \alpha \\ \cos \alpha \sin \beta \\ \cos \alpha \cos \beta \end{bmatrix}
Formula (LaTeX): S ^ { \prime } = \begin{bmatrix} S _ { H } \\ S _ { R } \end{bmatrix} = \begin{bmatrix} \cos \alpha \sin \beta \\ \cos \alpha \cos \beta \end{bmatrix}
Formula (LaTeX): [ \phi ] = \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos \phi & - \sin \phi \\ 0 & \sin \phi & \cos 

In [16]:
# 3. Access CodeBlocks and Formulas
for item, _ in doc.iterate_items():
    if isinstance(item, CodeItem):
        print(f"Code ({item.code_language}): {item.text}")
    elif isinstance(item, FormulaItem):
        print(f"Formula (LaTeX): {item.text}")


Formula (LaTeX): S = \begin{bmatrix} S _ { M } \\ S _ { E } \\ S _ { P } \end{bmatrix} = \begin{bmatrix} \cos \delta \cos \omega \\ - \cos \delta \sin \omega \\ \sin \delta \end{bmatrix}
Formula (LaTeX): \delta = \sin ^ { 1 } \{ 0 . 3 9 7 9 5 \cos \left [ 0 . 9 8 5 6 3 \left ( N - 1 7 3 \right ) \right ] \} \ \text {degrees} \right )
Formula (LaTeX): \omega = 1 5 ( t _ { s } - 1 2 ) \quad ( \text {degrees} )
Formula (LaTeX): [ \Phi ] = \begin{bmatrix} \cos \Phi & 0 & \sin \Phi \\ 0 & 1 & 0 \\ - \sin \Phi & 0 & \cos \Phi \end{bmatrix}
Formula (LaTeX): S ^ { \prime } = \begin{bmatrix} S _ { V } \\ S _ { H } \end{bmatrix} = \begin{bmatrix} \sin \alpha \\ \cos \alpha \sin \beta \\ \cos \alpha \cos \beta \end{bmatrix}
Formula (LaTeX): S ^ { \prime } = \begin{bmatrix} S _ { H } \\ S _ { R } \end{bmatrix} = \begin{bmatrix} \cos \alpha \sin \beta \\ \cos \alpha \cos \beta \end{bmatrix}
Formula (LaTeX): [ \phi ] = \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos \phi & - \sin \phi \\ 0 & \sin \phi & \cos 

In [17]:
from IPython.display import display, Math

for item, _ in doc.iterate_items():
    if isinstance(item, FormulaItem):
        display(Math(item.text))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [18]:
import re

def clean_latex(s):

    # Remove spaced digits: 0 . 3 9 7 9 5 -> 0.39795
    s = re.sub(r'(\d)\s*\.\s*(\d)', r'\1.\2', s)
    s = re.sub(r'(?<=\d)\s+(?=\d)', '', s)

    # Fix inverse trig
    s = re.sub(r'\\sin\s*\^\s*\{\s*1\s*\}', r'\\sin^{-1}', s)
    s = re.sub(r'\\cos\s*\^\s*\{\s*1\s*\}', r'\\cos^{-1}', s)
    s = re.sub(r'\\tan\s*\^\s*\{\s*1\s*\}', r'\\tan^{-1}', s)

    # Replace wrong braces
    s = s.replace(r'\{', '(')
    s = s.replace(r'\}', ')')

    # Remove dangling right delimiters
    s = s.replace(r'\right )', ')')

    return s

In [22]:
def fix_unbalanced_parentheses(s):

    stack = 0
    result = []

    for ch in s:
        if ch == '(':
            stack += 1
            result.append(ch)

        elif ch == ')':
            if stack > 0:
                stack -= 1
                result.append(ch)
            # skip extra )

        else:
            result.append(ch)

    # close missing parentheses
    result.extend(')' * stack)

    return ''.join(result)
def normalize_latex(s):

    s = s.replace(r'\left [', r'\left(')
    s = s.replace(r'\right ]', r'\right)')

    s = s.replace(r'\sin^{-1}', r'\arcsin')
    s = s.replace(r'\cos^{-1}', r'\arccos')
    s = s.replace(r'\tan^{-1}', r'\arctan')

    return s

In [23]:
from IPython.display import display, Math

for item, _ in doc.iterate_items():

    if isinstance(item, FormulaItem):

        # latex = clean_latex(item.text)
        latex = normalize_latex(clean_latex(item.text))

        try:
            latex = fix_unbalanced_parentheses(clean_latex(item.text))
            display(Math(latex))
        except Exception as e:
            print("Failed:", latex[:200])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [25]:
dict(doc).keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'field_regions', 'field_items', 'pages'])

In [28]:
for e in doc.iterate_items():
    print(e)

(SectionHeaderItem(self_ref='#/texts/1', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.SECTION_HEADER: 'section_header'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=56.699730000000045, t=458.0089760000001, r=128.95795800000005, b=449.13725692817684, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 15))], source=[], comments=[], orig='1. Introduction', text='1. Introduction', formatting=None, hyperlink=None, level=1), 1)
(TextItem(self_ref='#/texts/2', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=56.69999999999999, t=440.318, r=427.5828000000006, b=343.5077360780066, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 801))], source=[], comments=[], orig="Sun-tracking system plays an important role in the development of solar energy applic

In [35]:
list(doc.iterate_items())[32]

(FormulaItem(self_ref='#/texts/57', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.FORMULA: 'formula'>, prov=[ProvenanceItem(page_no=7, bbox=BoundingBox(l=192.78012719999998, t=543.0114575, r=427.52970000000005, b=502.2458615, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 92))], source=[], comments=[], orig='cos cos cos sin sin M E P S S S δ ω δ ω δ ⎡ ⎤ ⎡ ⎤ ⎢ ⎥ ⎢ ⎥ = = -⎢ ⎥ ⎢ ⎥ ⎢ ⎥ ⎢ ⎥ ⎣ ⎦ ⎣ ⎦ S (1)', text='S = \\begin{bmatrix} S _ { M } \\\\ S _ { E } \\\\ S _ { P } \\end{bmatrix} = \\begin{bmatrix} \\cos \\delta \\cos \\omega \\\\ - \\cos \\delta \\sin \\omega \\\\ \\sin \\delta \\end{bmatrix}', formatting=None, hyperlink=None),
 1)

In [36]:
doc.texts[57]

FormulaItem(self_ref='#/texts/57', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.FORMULA: 'formula'>, prov=[ProvenanceItem(page_no=7, bbox=BoundingBox(l=192.78012719999998, t=543.0114575, r=427.52970000000005, b=502.2458615, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 92))], source=[], comments=[], orig='cos cos cos sin sin M E P S S S δ ω δ ω δ ⎡ ⎤ ⎡ ⎤ ⎢ ⎥ ⎢ ⎥ = = -⎢ ⎥ ⎢ ⎥ ⎢ ⎥ ⎢ ⎥ ⎣ ⎦ ⎣ ⎦ S (1)', text='S = \\begin{bmatrix} S _ { M } \\\\ S _ { E } \\\\ S _ { P } \\end{bmatrix} = \\begin{bmatrix} \\cos \\delta \\cos \\omega \\\\ - \\cos \\delta \\sin \\omega \\\\ \\sin \\delta \\end{bmatrix}', formatting=None, hyperlink=None)

**COMMENT:**
- `FormulaItem` comes under `texts`

In [37]:
# Example usage
code_pdf_path = (
    "/mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/"
    "iid-platform/sample_data/named/"
    "pdf-with-code-blocks.pdf"
)
code_doc = load_docling_document(code_pdf_path)
code_fitz_pdf = fitz.open(code_pdf_path)

⚠️ Cache not found.
⏳ Converting PDF using Docling DocumentConverter...


[INFO] 2026-05-27 19:52:54,273 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 19:52:54,366 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 19:52:54,367 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 19:52:54,720 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 19:52:54,740 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-05-27 19:52:54,741 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer

✅ Conversion complete.
💾 Cache saved to: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/iid-platform/sample_data/named/pdf-with-code-blocks.docling.json

📄 Document Metadata
----------------------------------------
File Name      : pdf-with-code-blocks.pdf
Total Pages    : 15
Total Tables   : 4
Total Pictures : 22
Text Blocks    : 240
----------------------------------------


In [38]:
# 1. Configure the pipeline to enable code and formula detection
pipeline_options = PdfPipelineOptions()
pipeline_options.do_code_enrichment = True
pipeline_options.do_formula_enrichment = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# 2. Convert the document
result = converter.convert(code_pdf_path)
doc = result.document


[INFO] 2026-05-27 19:53:36,142 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 19:53:36,580 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 19:53:36,581 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-27 19:53:36,815 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-27 19:53:36,875 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-05-27 19:53:36,875 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer

In [39]:
# 3. Access CodeBlocks and Formulas
for item, _ in doc.iterate_items():
    if isinstance(item, CodeItem):
        print(f"Code ({item.code_language}): {item.text}")
    elif isinstance(item, FormulaItem):
        print(f"Formula (LaTeX): {item.text}")


Code (Python): 
Code (Python): from sentence_transformers import SentenceTransformer

        model=SentenceTransformer("sentence-transformers/al-MiniLM-L
        6-v2")
        embedding=model.encode(text_chunk)
Code (Python): You can also use lower-level transformers :

        from transformers import AutoTokenizer, AutoModel
        import torch

        tokenizer=AutoTokenizer.from_pretrained("sentence-transformer
        s/all-Mini-LM-L6-v2")
        model=AutoModel.from_pretrained("sentence-transformers/all-Mi
        niLM-L6-v2")

        inputs=tokenizer(text_chunk,return_tensors="pt",truncation=Tr
        ue,padding=True)
        outputs=model(**inputs)

        embedding=outputs.last_hidden_state.mean(dim=1)

    Pros:
Code (YAML): :o   Words: "I love cats"  ->  ["I", "love", "cats"]
  :o   Subwords or byte-pair encodings (BPE): "loveing" -
  :o   Characters: "cat"  ->  ["c", "a", "t"]

Code (YAML): :Original text: 10,000 words
  :Chunked into 500-word segments -> chunk

Cod

**COMMENTS:**
- Code block extraction also working